In [1]:
import torch
from dinosaw.utils import do_2D_pca
from dinosaw.wrappers import ModelTypes, MODEL_NAMES, get_models
from dinosaw.utils import do_2D_pca, get_features, add_custom_font

import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import minmax_scale, scale
from os import listdir

from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import font_manager


SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'alibi_coco_dinov2_s',)
models = get_models(selected_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

2026-07-24 13:48:53 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 13:48:53 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 13:48:53 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 13:48:53 | I | factory.py                 : 152 | Building wrapper 'alibi_coco_dinov2_s' on device cuda:0
2026-07-24 13:48:53 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=False, checkpoint_path='../../mod

In [8]:
def hide_axes(ax: plt.Axes):
    ax.set_xticks([])
    ax.set_yticks([])
    if hasattr(ax, 'set_zticks'):
        ax.set_zticks([])

In [9]:
def get_shared_pca(features_: list[np.ndarray], fg_masks: list[np.ndarray]) -> PCA:
    all_features = []
    for feature, fg_mask in zip(features_, fg_masks):
        masked = feature[fg_mask]
        all_features.append(masked)
        print(feature.shape, fg_mask.shape, masked.shape)
    all_features_concat = np.concatenate(all_features, axis=0)
    all_features_scaled = scale(all_features_concat, axis=0)
    pca = PCA(n_components=3)
    pca.fit(all_features_scaled)
    return pca

def apply_shared_pca(features: np.ndarray, fg_mask: np.ndarray, pca: PCA) -> np.ndarray:
    h, w, c = features.shape
    valid_features = features[fg_mask]
    features_scaled = scale(valid_features, axis=0)
    emb_3d = pca.transform(features_scaled)
    emb_3d_minmax = minmax_scale(emb_3d, feature_range=(0, 1), axis=0)

    out = np.zeros((h, w, 3), dtype=np.float32)
    out[fg_mask] = emb_3d_minmax
    return out

def _get(red_li, img_idx, ch, thr) -> np.ndarray:
    return red_li[img_idx][: , :, ch] > thr

In [10]:
images: list[Image.Image] = []
path = 'data/shared_pca/'
folder_name = 'bison'

for file_name in listdir(f'{path}/{folder_name}'):
    img = Image.open(f'{path}/{folder_name}/{file_name}').convert('RGB')

    shortest = min(img.size)
    L = 384
    sf = L / shortest
    # new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    new_size = (574, 384)
    img = img.resize(new_size, resample=Image.BILINEAR)

    images.append(img)

In [11]:
features = {k: [] for k in enabled_models}
features_reduced = {k: [] for k in enabled_models}
for img in images:
    for name, model in models.items():
        print(img.size)
        feat = get_features(model, img)
        reduced = do_2D_pca(feat, 3, pre_norm='std', post_norm='minmax')
        features[name].append(feat)
        features_reduced[name].append(reduced)

(574, 384)
2026-07-24 13:48:55 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:48:56 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:48:56 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:48:56 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:48:56 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:48:56 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:48:56 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:48:56 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:48:56 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2

In [12]:
# bison
red = features_reduced[enabled_models[-1]]
fg_masks =[_get(red, 0, 1, 0.6), ~_get(red, 1, 0, 0.3), ~_get(red, 2, 0, 0.3), ~_get(red, 3, 0, 0.4)]

selected_masks = fg_masks

In [13]:
%%capture
dv2_feats_tr = [f.transpose(1, 2, 0) for f in features[enabled_models[0]]]
dv2_shared_pca = get_shared_pca(dv2_feats_tr, fg_masks)

alibi_feats_tr = [f.transpose(1, 2, 0) for f in features[enabled_models[-1]]]
alibi_shared_pca = get_shared_pca(alibi_feats_tr, selected_masks)

dv2_fg_feats_reduced = [apply_shared_pca(feats, mask, dv2_shared_pca) for feats, mask in zip(dv2_feats_tr, selected_masks)]
alibi_fg_feats_reduced = [apply_shared_pca(feats, mask, alibi_shared_pca) for feats, mask in zip(alibi_feats_tr, selected_masks)]

In [14]:
images_: list[Image.Image] = []
path = 'data/shared_pca/'
folder_name = 'elephants'

for file_name in listdir(f'{path}/{folder_name}'):
    img = Image.open(f'{path}/{folder_name}/{file_name}').convert('RGB')

    shortest = min(img.size)
    L = 384
    sf = L / shortest
    # new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    new_size = (574, 384)
    img = img.resize(new_size, resample=Image.BILINEAR)

    images_.append(img)

In [15]:
features_ = {k: [] for k in enabled_models}
features_reduced_ = {k: [] for k in enabled_models}
for img in images_:
    for name, model in models.items():
        print(img.size)
        feat = get_features(model, img)
        reduced = do_2D_pca(feat, 3, pre_norm='std', post_norm='minmax')
        features_[name].append(feat)
        features_reduced_[name].append(reduced)

(574, 384)
2026-07-24 13:49:01 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:49:01 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:49:01 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:49:01 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:49:01 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:49:01 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:49:01 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2026-07-24 13:49:01 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,378,574] -> f: [1,384,27,41]
(574, 384)
2026-07-24 13:49:01 | I | wrapper.py                 :  92 | Processing image, size: [574, 384]
2

In [16]:
red = features_reduced_[enabled_models[0]]
fg_masks = [_get(red, 0, 0, 0.65), ~_get(red, 1, 0, 0.3), _get(red, 2, 0, 0.7), ~_get(red, 3, 0, 0.6)  ] 

selected_masks_ = fg_masks

In [17]:
%%capture
dv2_feats_tr = [f.transpose(1, 2, 0) for f in features_[enabled_models[0]]]
dv2_shared_pca = get_shared_pca(dv2_feats_tr, fg_masks)

alibi_feats_tr = [f.transpose(1, 2, 0) for f in features_[enabled_models[-1]]]
alibi_shared_pca = get_shared_pca(alibi_feats_tr, selected_masks_)

dv2_fg_feats_reduced_ = [apply_shared_pca(feats, mask, dv2_shared_pca) for feats, mask in zip(dv2_feats_tr, selected_masks_)]
alibi_fg_feats_reduced_ = [apply_shared_pca(feats, mask, alibi_shared_pca) for feats, mask in zip(alibi_feats_tr, selected_masks_)]

In [24]:
# %%capture
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')
n_rows, n_cols = len(selected_models) + 1, len(images)
W, H = 7.5, 2.3 * 1.5



fig, axs = plt.subplots(len(images), 6, figsize=(W , H ))

for i, (img, dv2_reduced, alibi_reduced, mask) in enumerate(zip(images, dv2_fg_feats_reduced, alibi_fg_feats_reduced, selected_masks)):
    axs[i, 0].imshow(img)
    hide_axes(axs[i, 0])

    axs[i, 1].imshow(dv2_reduced * mask[:, :, np.newaxis])
    hide_axes(axs[i, 1])
    

    axs[i, 2].imshow(alibi_reduced * mask[:, :, np.newaxis])
    hide_axes(axs[i, 2])

    if i == 0:
        axs[i, 1].set_title('DINOv2', )
        axs[i, 2].set_title('ALiBi-Dv2',  weight=700)


for i, (img, dv2_reduced, alibi_reduced, mask) in enumerate(zip(images_, dv2_fg_feats_reduced_, alibi_fg_feats_reduced_, selected_masks_)):
    axs[i, 3].imshow(img)
    hide_axes(axs[i, 3])

    axs[i, 4].imshow(dv2_reduced * mask[:, :, np.newaxis])
    hide_axes(axs[i, 4])
    

    axs[i, 5].imshow(alibi_reduced * mask[:, :, np.newaxis])
    hide_axes(axs[i, 5])

    if i == 0:
        axs[i, 4].set_title('DINOv2', )
        axs[i, 5].set_title('ALiBi-Dv2',  weight=700)


SAVE = True
if SAVE:
    plt.savefig("saved/S13.pdf", dpi=300, bbox_inches='tight')
    plt.close()


findfont: Failed to find font weight normal, now using 300.
